# 02 — Train ResNet18 on the spurious dataset
Settings come from `configs/resnet18_appendixE.yaml` and match MILAN's own `experiments/edit.py`: AdamW lr=1e-4, batch 128, early-stop on val loss with patience=4.

**Expected outcome:** clean validation accuracy lands around 85-95%; we'll measure adversarial-test accuracy separately and expect a clear drop, indicating the text shortcut took hold.

In [ ]:
import os, sys, yaml, torch
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
os.environ.setdefault('MILAN_DATA_DIR', str(Path.cwd().parent / 'data'))
os.environ.setdefault('MILAN_MODELS_DIR', str(Path.cwd().parent / 'models'))
from milan_repro.train.train_resnet18 import train

cfg = yaml.safe_load(open('../configs/resnet18_appendixE.yaml'))
version_dir = Path(os.environ['MILAN_DATA_DIR']) / 'imagenet-spurious-text' / '50pct'
ckpt = Path(os.environ['MILAN_MODELS_DIR']) / 'resnet18_spurious.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
info = train(cfg, version_dir, ckpt, device=device)
info['best_val_loss']

In [ ]:
# Evaluate clean and adversarial test accuracy.
from torch.utils.data import DataLoader
from milan_repro.milan_glue.register import load_trained
from milan_repro.data.spurious_dataset import load as load_spurious
from milan_repro.editing.evaluate import _accuracy

model = load_trained(ckpt, device=device)
_train_full, test_set = load_spurious(version_dir)
loader = DataLoader(test_set, batch_size=128, num_workers=2)
adv_acc = _accuracy(model, loader, device)
print(f'adversarial-test acc: {adv_acc:.4f}')